# Chapter 08 — The Project .pi Directory and Trust

**Companion to *Pi Agents*** · [`Pi Agents` chapter 08](https://programmer.ie/books/pi/08-chapter/)

| | |
|---|---|
| Chapter | `08-chapter.md` · weight 8 · `/books/pi/08-chapter/` |
| Notebook | `notebooks/pi/08-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Which project files are gated behind a trust decision, and **who wins when
several sources have an opinion?**

The ordering matters more than the gate. A repository cannot ask for trust, a
saved decision can be overridden by a flag, and print mode — which has nobody to
ask — quietly behaves as "skip".


## What this notebook establishes

- Nine documented project paths require a decision: the seven under `.pi/`, plus `.agents/skills/.../SKILL.md`. A bare `.pi/` directory does **not**, and neither do `AGENTS.md` / `CLAUDE.md`.
- `AGENTS.md` loads **regardless of trust**. Declining is not isolation: the extension and the prompt template stay out, the context file comes in.
- The decision order is **flag → extension handler → saved decision → `defaultProjectTrust`**, and an extension that answers `undecided` hands the decision on rather than making it.
- A saved decision for a directory covers its children; a child decision overrides without rewriting the parent, and setting a decision to `null` clears it.

## What this notebook does **not** establish

- **Print mode cannot ask.** With no UI and no override, `ask` behaves as skip. Every ordering row below is therefore observed through a headless driver, which is exactly the situation the chapter warns about.
- **`remember` is declared and not tested**, by the chapter's own account.
- **No isolation method is evaluated.** `metadata/08-chapter.yaml` records that evaluating a container, VM or credential proxy is out of scope — that is `docs/containerization.md` and chapter 42.
- **The `sessionDir` setting is read before trust resolves**, so it is the one thing a declined trust does not undo. That is chapter 5's evidence, run there.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — Pi's own trust predicates against real directories, and the shipped binary in print mode for the decision order.
* **Evidence scope:** `in_process_runtime` for the predicates and the store; `shipped_binary` for the decision order.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(8))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/08-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Experiment 1: which paths are gated

`hasTrustRequiringProjectResources` is Pi's own predicate, pointed at real temporary
directories each containing exactly one candidate path. The last two rows are the
exceptions that matter: a bare `.pi/`, and context files.

In [3]:
TRUST = pinb.driver_source("ch08-trust.ts")
trust = pinb.driver(TRUST, label="ch08-trust", timeout=300)
print(pinb.md_table(
    ["project path", "requires a decision?"],
    [[t["path"], "yes" if t["requiresDecision"] else "no"] for t in trust["triggers"]]
    + [[trust["withContextFiles"]["path"] + "  (the exception)", "yes" if trust["withContextFiles"]["requiresDecision"] else "no"],
       [trust["ancestor"]["path"], "yes" if trust["ancestor"]["requiresDecision"] else "no"]],
))

| project path | requires a decision? |
|---|---|
| .pi/settings.json | yes |
| .pi/mcp.json | yes |
| .pi/extensions/example.ts | yes |
| .pi/skills/example/SKILL.md | yes |
| .pi/prompts/example.md | yes |
| .pi/themes/example.json | yes |
| .pi/SYSTEM.md | yes |
| .pi/APPEND_SYSTEM.md | yes |
| .agents/skills/example/SKILL.md | yes |
| .pi/ | no |
| AGENTS.md | no |
| AGENTS.md + CLAUDE.md only  (the exception) | no |
| start below a trigger in an ancestor | yes |


In [4]:
gated = {t["path"] for t in trust["triggers"] if t["requiresDecision"]}
ungated = {t["path"] for t in trust["triggers"] if not t["requiresDecision"]}
expected_gated = {
    ".pi/settings.json", ".pi/mcp.json", ".pi/extensions/example.ts",
    ".pi/skills/example/SKILL.md", ".pi/prompts/example.md", ".pi/themes/example.json",
    ".pi/SYSTEM.md", ".pi/APPEND_SYSTEM.md", ".agents/skills/example/SKILL.md",
}

pinb.show_checks([
    pinb.check("exactly the nine documented paths are gated",
               gated == expected_gated, f"{len(gated)} gated"),
    pinb.check("a bare .pi/ directory needs no decision", ".pi/" in ungated, str(sorted(ungated))),
    pinb.check("a context file needs no decision", "AGENTS.md" in ungated, "AGENTS.md is ungated"),
    pinb.check("AGENTS.md + CLAUDE.md together still need no decision",
               not trust["withContextFiles"]["requiresDecision"], "no"),
    pinb.check("a trigger in an ancestor counts when you start below it",
               trust["ancestor"]["requiresDecision"], "yes"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  ---------------------
PASS  exactly the nine documented paths are gated  9 gated
PASS  a bare .pi/ directory needs no decision  ['.pi/', 'AGENTS.md']
PASS  a context file needs no decision  AGENTS.md is ungated
PASS  AGENTS.md + CLAUDE.md together still need no decision  no
PASS  a trigger in an ancestor counts when you start below it  yes

5/5 assertions held.


## Experiment 2: saved decisions are inherited, and clearing works

`trust.json` in the agent directory is the durable record. The sequence below is
three `set` calls and four reads, which is the whole inheritance rule.

In [5]:
print(pinb.md_table(
    ["after", "read for the repository", "read for repo/src"],
    [["nothing saved", trust["inheritance"]["initial"], "-"],
     ["parent set to yes", trust["inheritance"]["afterParentYes"]["repo"], trust["inheritance"]["afterParentYes"]["child"]],
     ["child set to no", trust["inheritance"]["afterChildNo"]["repo"], trust["inheritance"]["afterChildNo"]["child"]],
     ["parent cleared with null", trust["inheritance"]["afterCleared"]["repo"], "-"]],
))

| after | read for the repository | read for repo/src |
|---|---|---|
| nothing saved |  | - |
| parent set to yes | True | True |
| child set to no | True | False |
| parent cleared with null |  | - |


Row three is the one to notice: the child says **no** and the parent still
says **yes**. The store resolves by walking up and taking the nearest decision, so
refusing a subdirectory does not poison its parent.

## Experiment 3: who decides, in what order

Every row is a real `pi --print` run against the shipped binary. The project
extension writes a flag file when its factory runs, which is the observable: did the
protected project resource load, yes or no.

In [6]:
DECISION = pinb.driver_source("ch08-decision.ts")
order = pinb.driver(DECISION, label="ch08-decision", timeout=420)
print(pinb.md_table(
    ["who has an opinion", "the project extension loaded?"],
    [[r["who"], "YES" if r["loaded"] else "no"] for r in order],
))

| who has an opinion | the project extension loaded? |
|---|---|
| nothing at all | no |
| --approve | YES |
| --no-approve | no |
| defaultProjectTrust: always | YES |
| defaultProjectTrust: ask | no |
| defaultProjectTrust: never | no |
| saved yes | YES |
| saved no | no |
| saved yes + defaultProjectTrust: never | YES |
| saved no + --approve | YES |
| extension says yes + saved no | YES |
| extension says no + saved yes | no |
| extension says undecided + saved yes | YES |
| extension says yes + --no-approve | no |


In [7]:
loaded = {r["who"]: r["loaded"] for r in order}

pinb.show_checks([
    pinb.check("nothing configured: print mode cannot ask, so it skips",
               loaded["nothing at all"] is False, "not loaded"),
    pinb.check("--approve loads, --no-approve does not",
               loaded["--approve"] is True and loaded["--no-approve"] is False, "flag decides first"),
    pinb.check("defaultProjectTrust: always loads; ask and never skip",
               loaded["defaultProjectTrust: always"] is True
               and not loaded["defaultProjectTrust: ask"]
               and not loaded["defaultProjectTrust: never"], "setting decides last"),
    pinb.check("a saved decision beats the default setting",
               loaded["saved yes + defaultProjectTrust: never"] is True, "saved yes wins over never"),
    pinb.check("a command-line flag beats a saved decision",
               loaded["saved no + --approve"] is True, "--approve over a saved no"),
    pinb.check("an extension handler beats a saved decision",
               loaded["extension says yes + saved no"] is True
               and loaded["extension says no + saved yes"] is False, "handler decides before the store"),
    pinb.check("an extension that answers undecided hands the decision on",
               loaded["extension says undecided + saved yes"] is True, "the saved yes applied"),
    pinb.check("the flag still outranks an extension",
               loaded["extension says yes + --no-approve"] is False, "--no-approve wins"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------  --------------------------------
PASS  nothing configured: print mode cannot ask, so it skips  not loaded
PASS  --approve loads, --no-approve does not  flag decides first
PASS  defaultProjectTrust: always loads; ask and never skip  setting decides last
PASS  a saved decision beats the default setting  saved yes wins over never
PASS  a command-line flag beats a saved decision  --approve over a saved no
PASS  an extension handler beats a saved decision  handler decides before the store
PASS  an extension that answers undecided hands the decision on  the saved yes applied
PASS  the flag still outranks an extension  --no-approve wins

8/8 assertions held.


Read the table top to bottom and the order is forced: every source that
appears **later** in this list is contradicted by an earlier one.

```
command-line flag  >  extension project_trust handler  >  saved decision  >  defaultProjectTrust
```

That is a design choice, not an accident of implementation, and it has a
consequence worth stating: **a hostile repository cannot grant itself trust**, and
neither can a project file — `defaultProjectTrust` only reads from the agent
directory.

### The ungated channel

The exception in experiment 1 is the part to carry forward. `AGENTS.md` and
`CLAUDE.md` load regardless of trust. Declining a project stops its extensions,
prompts, skills, themes and system-prompt files; it does **not** stop its
instructions from being read into the model's context. `-nc` / `--no-context-files`
is the only documented way to remove that channel, and it is a per-run choice.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [8]:
PATTERNS = ['ch08-trust/trust.test.ts', 'ch08-trust/decision.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch08-trust/trust.test.ts, ch08-trust/decision.test.ts', show="AGENTS.md")

$ node --test ch08-trust/trust.test.ts, ch08-trust/decision.test.ts
  PASS  18 passed, 0 failed, 0 skipped  in 6235 ms

  tests matching 'AGENTS.md':
    ok   untrusted: the project's extension and prompt template do not load, but its AGENTS.md still does


## Your turn: predict, then run

**Predict first.** With **no** flag, **no** extension and **no** saved decision, in
`--print` mode: does a project `AGENTS.md` load? Does a project `.pi/settings.json`?

**Then change one input.** `ORDER` below adds one row to the same matrix: a saved
**no** plus an extension that answers `undecided`. Which way should it fall?

**Predict the boundary.** What would an extension have to return to *not* be
consulted at all? (The chapter records that returning the bare string `"yes"` is
silently ignored, because the handler returns an object.)

In [9]:
expected = {
    "undecided + saved no": False,
    "flag beats everything": False,
}
print("predicted:")
for k, v in expected.items():
    print(f"  {k:26s} -> {v}")
print()
print("already observed above:")
print(f"  {'undecided + saved yes':26s} -> {loaded['extension says undecided + saved yes']}")
print(f"  {'yes + --no-approve':26s} -> {loaded['extension says yes + --no-approve']}")

assert loaded["extension says undecided + saved yes"] is True
assert loaded["extension says yes + --no-approve"] is False
print()
print("held: undecided defers to the store, and the flag outranks every extension")

predicted:
  undecided + saved no       -> False
  flag beats everything      -> False

already observed above:
  undecided + saved yes      -> True
  yes + --no-approve         -> False

held: undecided defers to the store, and the flag outranks every extension


## Interpretation

Trust is a **gate on loading**, decided in a fixed order, with exactly one ungated
channel. Three practical consequences:

1. **Decide in scripts, not in prompts.** Print mode cannot ask. `--approve` and
   `--no-approve` are the only two answers available to a batch job, and the chapter
   is explicit that `always` loads while `ask` and `never` skip there.
2. **Declining is not isolation.** It removes executable and configurable project
   material. It does not remove instructions. If that matters, turn off context
   files for the run.
3. **A remembered decision is sticky in one direction.** It can be overridden by a
   flag or a handler in every run, but it persists in `trust.json` until someone
   clears it. Chapter 42 is about what "trusted" does and does not buy you.

## Sources, execution mode and limitations

**Execution mode:** **Run** — Pi's trust predicate against eleven real directory layouts, `ProjectTrustStore` across four reads, and fourteen headless runs of the shipped binary.

**Evidence scope:** `in_process_runtime` plus `shipped_binary`. The provider is scripted; the gate, the store and the decision order are Pi's.

### What was read or run

- **Ran** `hasTrustRequiringProjectResources` on eleven temporary directories, each holding exactly one candidate path, plus an `AGENTS.md + CLAUDE.md` layout and a start-below-an-ancestor case.
- **Ran** four reads against one `ProjectTrustStore` across three `set` calls.
- **Ran** fourteen `pi --print` invocations, offline, each in a fresh temporary working directory, with the flag / saved decision / setting / extension handler varied.
- **Ran** `ch08-trust/trust.test.ts` and `ch08-trust/decision.test.ts`.
- **Read** `examples/evidence.json`, chapter 8 rows (18 claims).

### Limitations

- **Print mode cannot ask.** Every ordering row is therefore observed in the mode where the answer is forced. A terminal session with a person present would offer a prompt instead, which changes `ask` from skip to ask — a behaviour that cannot be exercised here.
- **The ungated-channel claim is structural.** The table shows that `AGENTS.md` does not require a decision; the chapter's own test (`untrusted: the project's extension and prompt template do not load, but its AGENTS.md still does`) is the evidence that it still loads, and it ran above.
- **`remember` was not exercised**, by the chapter's own account. No isolation method is evaluated here; that is chapter 42 and it is mostly *not run*.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
